# Atlas Observatory — TQQQ / QQQ descriptive monitor

**Data.** Public daily adjusted prices and volume via `yfinance`.

The proxy is `1 - rolling correlation`, a pairwise correlation-distance measure, distinct from cross-sectional return dispersion. Zones are descriptive threshold labels, with no predictive validation.

Dispersion thresholds use expanding observations strictly before each labeled date (minimum 20 valid proxy observations). Ties can leave some buckets empty; all four buckets are not guaranteed. The first observations remain uncalibrated. Current-year returns are measured from the prior year's final available close, using pre-year data for rolling-window warm-up.

**Verified snapshot.** The metrics table has latest observation 2026-10-13. It was calculated in the verified Windows/Python 3.12.1 run and replayed identically offline; the current HTML was later re-rendered from that same table with the revised layout (see `results/atlas_layout_validation.json`). Notebook execution outputs remain cleared; the exported metrics table and manifest record the calculation. The HTML does not update itself.
**Presentation update.** The six charts use separate scales and short reading guides. The current-state chart retains faint historical observations, a teal twenty-observation trail, and a highlighted latest point. Date-range and reset controls change the view only. The footer identifies daily data, the latest market observation, and the saved snapshot-generation time in America/Los_Angeles. Metrics remain unchanged from the verified calculation.


In [ ]:
# If needed, uncomment:
# !pip install yfinance plotly pandas numpy

import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import sys
from pathlib import Path
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").is_file())
sys.path.insert(0, str(REPO_ROOT))
from src.vdcloud import adjusted_close, select_field, price_returns, prior_quantile_buckets, classify_zone
from src.repro import download


In [ ]:
target = "TQQQ"
benchmark = "QQQ"
year_start = pd.Timestamp(datetime.now().year, 1, 1)
start_date = (year_start - pd.Timedelta(days=180)).strftime("%Y-%m-%d")
end_date = None             # None = latest available -- this notebook is meant to be re-run fresh

window = 20
annualization = np.sqrt(252)

In [ ]:
tickers = [target, benchmark]

raw = download(
    tickers,
    start=start_date,
    end=end_date,
    auto_adjust=True,
    progress=False
)

prices = adjusted_close(raw, tickers)
volume = select_field(raw, ("Volume",), tickers)

returns = price_returns(prices)

prices.tail()

In [ ]:
# Normalized price
prior_close = prices.loc[prices.index < year_start].dropna().iloc[-1]
norm_price = prices / prior_close * 100

# YTD return
ytd_return = (prices[target].iloc[-1] / prior_close[target] - 1)

# Rolling realized volatility
realized_vol = returns[target].rolling(window).std() * annualization

# Rolling correlation vs QQQ
rolling_corr = returns[target].rolling(window).corr(returns[benchmark])

# Simple dispersion proxy
# Higher = less tied to QQQ / more fragmented from benchmark movement
dispersion_proxy = 1 - rolling_corr

# Rolling average volume
rolling_volume = volume[target].rolling(window).mean()

metrics = pd.DataFrame({
    "Price_Index": norm_price[target],
    "Realized_Vol": realized_vol,
    "Correlation_vs_QQQ": rolling_corr,
    "Dispersion_Proxy": dispersion_proxy,
    "Rolling_Volume": rolling_volume
}).dropna()
metrics = metrics.loc[metrics.index >= year_start].copy()
if metrics.empty:
    raise ValueError("No current-year observations with complete rolling metrics.")

metrics.tail()

In [ ]:
# Each historical label uses only prior observed proxy values in this year.
disp_high_threshold = metrics["Dispersion_Proxy"].shift(1).expanding(min_periods=20).quantile(.75)
metrics["Zone"] = [classify_zone(v, d, threshold) for v, d, threshold in
                   zip(metrics["Realized_Vol"], metrics["Dispersion_Proxy"], disp_high_threshold)]
metrics["Vol_Bucket"] = pd.cut(metrics["Realized_Vol"], bins=[0, .40, .70, 1., np.inf],
                              labels=["Low", "Medium", "High", "Very High"], include_lowest=True)
metrics["Disp_Bucket"] = prior_quantile_buckets(metrics["Dispersion_Proxy"],
                                              ["Very Low", "Low", "Medium", "High"], min_periods=20)
metrics["Disp_Bucket"] = pd.Categorical(metrics["Disp_Bucket"], categories=["Very Low", "Low", "Medium", "High"], ordered=True)
print("Dispersion buckets (prior-data thresholds; initial observations uncalibrated):")
print(metrics["Disp_Bucket"].value_counts())
metrics.tail()


In [ ]:
# Presentation only; the metric calculations above are unchanged.
from src.atlas_dashboard import build_charts, render_dashboard

latest = metrics.iloc[-1]
latest_date = metrics.index[-1].strftime("%d %b %Y")
charts = build_charts(metrics, target, benchmark)


In [ ]:
print(f"Atlas snapshot: {target} vs {benchmark}, latest observation {latest_date}")
print(f"YTD adjusted return from prior-year close: {ytd_return:.2%}")
print(f"20D annualized volatility: {latest['Realized_Vol']:.2%}")
print(f"Correlation distance: {latest['Dispersion_Proxy']:.6f}")
print(f"Descriptive zone: {latest['Zone']}")
print("Thresholds use prior observations; zones have no predictive validation.")


In [ ]:
# Self-contained, responsive HTML with captions and explicit data freshness.
# A saved generation timestamp makes identical cached replays byte-identical.
render_dashboard(metrics, target, benchmark, ytd_return,
                 REPO_ROOT / "atlas" / "index.html",
                 snapshot_path=REPO_ROOT / "results" / "atlas_dashboard_snapshot.json")
print("Saved dashboard to atlas/index.html")
